# (b) Train YOLO-seg từ dataset của `auto_label.ipynb`

Kaggle: bật **GPU** và **Internet**. **Add Input → Your Work → Notebooks →** notebook đã chạy `auto_label.ipynb`. Chạy lại auto-label thì vào Input cập nhật version.

Test chỉ có nhãn **flood thật**, nên khi đánh giá test chỉ đọc dòng `flood`. Output: `best.pt`, `water_flood/` (log, biểu đồ), `loss.png`.

In [ ]:
%pip -q install ultralytics

In [ ]:
import shutil
from pathlib import Path
import yaml
import pandas as pd
import torch
from matplotlib import pyplot as plt
from ultralytics import YOLO

# ============================================================
# CONFIG
# ============================================================
# Dataset = output của auto_label.ipynb (Add Input -> Your Work -> Notebooks -> notebook auto-label).
DATA_DIR = next(Path("/kaggle/input").rglob("dataset/data.yaml")).parent
MODEL_NAME = "yolo11n-seg.pt"   # đổi "yolo11s-seg.pt" nếu muốn chính xác hơn
EPOCHS = 100
IMG_SIZE = 640
BATCH_SIZE = 32                 # ~1000 ảnh: batch 128 chỉ còn ~8 bước/epoch, 32 hợp lý hơn
LR = 1e-4
DEVICE = [0, 1] if torch.cuda.device_count() > 1 else (0 if torch.cuda.is_available() else "cpu")

RUN_DIR = Path("/kaggle/working/water_flood")
CHECKPOINT_PATH = RUN_DIR / "weights" / "last.pt"

print("Dataset:", DATA_DIR)
for name in ("READY_FOR_TRAINING.txt", "NOT_READY_FOR_TRAINING.txt"):
    if (DATA_DIR / name).exists():
        print(f"--- {name} ---\n" + (DATA_DIR / name).read_text())

# data.yaml gốc trỏ tới /kaggle/working/dataset của notebook auto-label -> ghi lại với đường dẫn thật
cfg = yaml.safe_load((DATA_DIR / "data.yaml").read_text())
cfg["path"] = str(DATA_DIR)
DATA_YAML = "/kaggle/working/data.yaml"
Path(DATA_YAML).write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))

# ============================================================
# 1. TRAIN / RESUME
# ============================================================
if CHECKPOINT_PATH.exists():
    print(f"--> Train tiếp từ checkpoint: {CHECKPOINT_PATH}")
    model = YOLO(CHECKPOINT_PATH)
    model.train(resume=True)
else:
    print("--> Train mới | Device:", DEVICE)
    model = YOLO(MODEL_NAME)
    model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        optimizer="Adam",
        lr0=LR,
        workers=4,
        project="/kaggle/working",
        name="water_flood",
        exist_ok=True,
        pretrained=True,
        cache=True,
        plots=True,
    )

# ============================================================
# 2. ĐÁNH GIÁ (val: mọi class; test: chỉ có nhãn flood thật -> chỉ đọc dòng flood)
# ============================================================
best = YOLO(RUN_DIR / "weights" / "best.pt")
dev = 0 if torch.cuda.is_available() else "cpu"
print("\n--> VAL")
best.val(data=DATA_YAML, split="val", imgsz=IMG_SIZE, batch=BATCH_SIZE, device=dev)
print("\n--> TEST (chỉ flood)")
best.val(data=DATA_YAML, split="test", imgsz=IMG_SIZE, batch=BATCH_SIZE, device=dev)

# ============================================================
# 3. COPY BEST.PT RA /kaggle/working/
# ============================================================
shutil.copy(RUN_DIR / "weights" / "best.pt", "/kaggle/working/best.pt")
print("\n--> Đã copy: /kaggle/working/best.pt")

# ============================================================
# 4. PLOT LOSS
# ============================================================
df = pd.read_csv(RUN_DIR / "results.csv")
df.columns = df.columns.str.strip()
plt.figure(figsize=(10, 6))
for col in ("train/box_loss", "train/seg_loss", "train/cls_loss", "train/dfl_loss"):
    if col in df.columns:
        plt.plot(df["epoch"], df[col], label=col.split("/")[1])
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training Loss"); plt.legend(); plt.grid(True)
plt.savefig("/kaggle/working/loss.png")
plt.show()